# Day 02 · Python that senior engineers write

**Goal:** write the six constructs that turn a script into a library — **classes with dunder methods** (a PyTorch-shaped `Dataset`), **generators** (stream a file bigger than RAM), **decorators** (`@timer`, `@retry`, `@lru_cache`), **context managers**, **dataclasses** and **type hints** — and see *why* each one matters for AI code.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`. Then *Runtime → Run all*.

**Runtime:** CPU is fine.

| Section | What you'll build |
|---|---|
| 1 | A class with `__init__`, `__len__`, `__getitem__`, `__repr__`, `__call__` → `ReviewDataset` |
| 2 | Inheritance, `super()`, composition |
| 3 | Iterators & generators → a streaming pipeline in constant memory |
| 4 | Decorators: `@timer`, `@retry`, `@lru_cache`, `*args/**kwargs` |
| 5 | Context managers (`with`), `contextlib` |
| 6 | Dataclasses, type hints, `functools`, `collections` |
| 7 | 🎯 Try it yourself + Homework starter |

## 0. Setup
Standard library only again. We print versions and set a seed so every run looks the same.

In [3]:
import sys, time, random, functools, itertools, tracemalloc
from collections import Counter, defaultdict, namedtuple, deque
from dataclasses import dataclass, field
from contextlib import contextmanager
random.seed(42)
print("Python", sys.version.split()[0])

Python 3.13.15


## 1. Classes and the dunder hooks Python calls for you 🚂

**Analogy:** an IRCTC coach. The *class* is the blueprint ("72 berths, a chart on the door"); coach S4 on today's Rajdhani is an *instance*. `len(coach)` asks how many berths; `coach[23]` asks who is in berth 23. You never call `__len__` yourself — you write `len(x)` and Python looks for the hook.

That is the whole idea of **dunder** (double-underscore) methods: *you implement the hook, Python's syntax calls it.* PyTorch's `DataLoader` is the ticket examiner: it calls `len(ds)` once and `ds[i]` for each sample.

In [4]:
@dataclass(frozen=True)          # a tiny immutable record (more on dataclasses in section 6)
class Sample:
    text: str
    label: int                   # 1 = positive, 0 = negative

class ReviewDataset:
    """A PyTorch-shaped dataset: __len__ + __getitem__ is all DataLoader needs."""

    def __init__(self, rows: list[Sample], transform=None):
        self.rows = rows
        self.transform = transform          # optional function applied to each text

    def __len__(self) -> int:               # makes len(ds) work
        return len(self.rows)

    def __getitem__(self, i: int) -> tuple[str, int]:   # makes ds[i] work
        if isinstance(i, slice):                         # ds[1:3] -> a list of samples
            return [self[j] for j in range(*i.indices(len(self)))]
        s = self.rows[i]
        text = self.transform(s.text) if self.transform else s.text
        return text, s.label

    def __repr__(self) -> str:              # what you see when you print it
        return f"ReviewDataset(n={len(self)}, transform={getattr(self.transform, '__name__', None)})"

rows = [
    Sample("The biryani was SUPERB", 1), Sample("Cold and late, never again", 0),
    Sample("Decent chai, great vibe", 1),  Sample("Overpriced for what it is", 0),
    Sample("Loved the dosa!", 1),          Sample("Rude staff", 0),
]
ds = ReviewDataset(rows, transform=str.lower)
print(ds)
print("len(ds) =", len(ds))
print("ds[0]   =", ds[0])
print("ds[-1]  =", ds[-1])                 # negative indexing works because the list handles it

ReviewDataset(n=6, transform=lower)
len(ds) = 6
ds[0]   = ('the biryani was superb', 1)
ds[-1]  = ('rude staff', 0)


In [5]:
# Because __getitem__ raises IndexError at the end, Python can even iterate it with a for-loop:
for text, label in ds:
    print(f"{label} | {text}")

# Slicing works because we handle the slice object explicitly:
print(ds[1:3])

1 | the biryani was superb
0 | cold and late, never again
1 | decent chai, great vibe
0 | overpriced for what it is
1 | loved the dosa!
0 | rude staff
[('cold and late, never again', 0), ('decent chai, great vibe', 1)]


> 💡 **On Day 21** the only changes will be `class ReviewDataset(torch.utils.data.Dataset)` and returning tensors instead of strings. The `DataLoader` calls exactly these two methods.

### `__call__`: an instance that behaves like a function
This is why in PyTorch you write `model(x)` and not `model.forward(x)`. The object *is* callable.

In [6]:
class Tokenizer:
    """Maps words -> integer ids. Calling the instance tokenizes."""
    def __init__(self, texts):
        vocab = sorted({w for t in texts for w in t.lower().split()})
        self.word2id = {w: i + 1 for i, w in enumerate(vocab)}     # 0 reserved for <unk>

    def __call__(self, text: str) -> list[int]:
        return [self.word2id.get(w, 0) for w in text.lower().split()]

    def __len__(self):
        return len(self.word2id) + 1

tok = Tokenizer([s.text for s in rows])
print("vocab size:", len(tok))
print(tok("the biryani was cold"))          # 'cold' is in vocab, so it gets an id; unknown words -> 0
print(tok("the pizza was cold"))            # 'pizza' unseen -> 0

vocab size: 23
[19, 3, 21, 5]
[19, 0, 21, 5]


## 2. Inheritance and `super()`: extend, don't copy 🧬

A child class gets everything the parent has and can override pieces. Inside the child's `__init__`, `super().__init__()` runs the parent's setup **first** — forgetting it in PyTorch's `nn.Module` is a classic bug (parameters never get registered).

Rule of thumb: prefer **composition** ("has a") over deep inheritance ("is a").

In [7]:
class BaseModel:
    def __init__(self, name: str):
        self.name = name
        self.history = []                          # every model records its losses

    def fit(self, data):
        raise NotImplementedError("subclasses must implement fit")

    def __repr__(self):
        return f"{type(self).__name__}(name={self.name!r}, epochs_run={len(self.history)})"

class MeanModel(BaseModel):
    """Predicts the mean of the training targets. The dumbest baseline — always build it first."""
    def __init__(self, name="mean-baseline"):
        super().__init__(name)                     # parent's setup first
        self.mean_ = None

    def fit(self, data):
        self.mean_ = sum(data) / len(data)
        self.history.append(0.0)
        return self                                # returning self lets you chain: MeanModel().fit(x).predict()

    def predict(self, n=1):
        return [self.mean_] * n

m = MeanModel().fit([10, 20, 30, 40])
print(m, "->", m.predict(3))
print("isinstance(m, BaseModel):", isinstance(m, BaseModel))

try:
    BaseModel("x").fit([1])
except NotImplementedError as e:
    print("NotImplementedError:", e)

MeanModel(name='mean-baseline', epochs_run=1) -> [25.0, 25.0, 25.0]
isinstance(m, BaseModel): True
NotImplementedError: subclasses must implement fit


## 3. Iterators and generators: a function that can pause 🍛

**Analogy:** a *list* is a thali — the kitchen prepares all 12 items and hands over one full plate. If the plate is too big for the table (RAM), nothing arrives. A *generator* is the dosa counter — you ask, one dosa comes off the tawa, you eat, you ask again. `yield` is the moment the cook hands you the dosa and **pauses**, keeping the tawa hot.

First, the mechanism: `iter()` and `next()`.

In [8]:
nums = [10, 20, 30]
it = iter(nums)                 # a for-loop does this...
print(next(it), next(it), next(it))
try:
    next(it)                    # ...and stops when it sees this
except StopIteration:
    print("StopIteration -> the for-loop ends here")

def read_rows():
    """A generator: the body does not run until someone calls next()."""
    print("  (opening file...)")
    for line in ["1200,Pune", "85.5,Delhi", "499,Mumbai"]:
        amt, city = line.split(",")
        print(f"  (parsed {line})")
        yield float(amt), city              # hand over, PAUSE here
    print("  (file closed)")

g = read_rows()
print("generator created, nothing printed yet:", g)
print("next ->", next(g))
print("next ->", next(g))
print("remaining via list():", list(g))     # drains the rest, then the function finishes

10 20 30
StopIteration -> the for-loop ends here
generator created, nothing printed yet: <generator object read_rows at 0x7f61708b3a60>
  (opening file...)
  (parsed 1200,Pune)
next -> (1200.0, 'Pune')
  (parsed 85.5,Delhi)
next -> (85.5, 'Delhi')
  (parsed 499,Mumbai)
  (file closed)
remaining via list(): [(499.0, 'Mumbai')]


In [ ]:
# The number-one generator bug: they are single-use.
squares = (x * x for x in range(5))         # generator expression = comprehension with ( )
print(list(squares))
print(list(squares), "<- empty! already exhausted")

### Memory: list vs generator, measured
`tracemalloc` tracks allocations. Building a list of a million squares needs ~40 MB; the generator object is ~200 bytes no matter how many items it will produce.

In [9]:
N = 1_000_000
tracemalloc.start()
squares_list = [i * i for i in range(N)]
_, peak_list = tracemalloc.get_traced_memory(); tracemalloc.stop()

tracemalloc.start()
squares_gen = (i * i for i in range(N))
total = sum(squares_gen)                    # consumes the stream, one item alive at a time
_, peak_gen = tracemalloc.get_traced_memory(); tracemalloc.stop()

print(f"list      peak memory: {peak_list/1e6:8.2f} MB")
print(f"generator peak memory: {peak_gen/1e6:8.4f} MB   (sum = {total:,})")
print(f"generator object itself: {sys.getsizeof((i for i in range(N)))} bytes")

list      peak memory:    40.47 MB
generator peak memory:   0.0305 MB   (sum = 333,332,833,333,500,000)
generator object itself: 192 bytes


### A streaming pipeline: file → parse → batch, in constant memory
Each stage **pulls** from the previous one. At any moment at most one batch is alive. This is the shape of every data pipeline — PyTorch `DataLoader`, Hugging Face `streaming=True`, Spark — and you can write it in 15 lines.

We first write a 200k-row CSV of fake UPI transactions *using a generator* (never holding it in a list), then stream it back.

In [10]:
def fake_transactions(n):
    """Yields n (amount, city) pairs lazily."""
    cities = ["Pune", "Delhi", "Mumbai", "Kochi", "Jaipur", "Bengaluru"]
    for _ in range(n):
        yield round(random.lognormvariate(6, 1), 2), random.choice(cities)

with open("upi.csv", "w") as f:
    f.write("amount,city\n")
    for amt, city in fake_transactions(200_000):        # streams straight to disk
        f.write(f"{amt},{city}\n")

import os
print(f"upi.csv: {os.path.getsize('upi.csv')/1e6:.1f} MB on disk")

upi.csv: 2.8 MB on disk


In [11]:
def read_lines(path):                       # stage 1: one line at a time from disk
    with open(path, encoding="utf-8") as f:
        next(f)                                 # skip header
        for line in f:
            yield line.rstrip("\n")

def parse(lines):                               # stage 2: str -> (float, str)
    for line in lines:
        amt, city = line.split(",")
        yield float(amt), city

def batched(items, size):                       # stage 3: group into lists of `size`
    batch = []
    for it in items:
        batch.append(it)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch                             # last partial batch

tracemalloc.start()
pipeline = batched(parse(read_lines("upi.csv")), size=1000)   # nothing has run yet!
total, count, biggest, per_city = 0.0, 0, 0.0, defaultdict(float)
for batch in pipeline:                          # now data flows, 1000 rows at a time
    for amt, city in batch:
        total += amt; count += 1; biggest = max(biggest, amt); per_city[city] += amt
_, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()

print(f"rows={count:,}  mean=₹{total/count:,.2f}  max=₹{biggest:,.2f}  peak RAM={peak/1e6:.2f} MB")
print("top city by volume:", max(per_city.items(), key=lambda kv: kv[1]))

# Python 3.12+ has itertools.batched built in; on 3.11 use ours. Either way: same idea.
print("itertools.batched available:", hasattr(itertools, "batched"))

rows=200,000  mean=₹662.61  max=₹27,154.37  peak RAM=0.19 MB
top city by volume: ('Kochi', 22497147.47000001)
itertools.batched available: True


> 🧠 **Interview Q:** *"What is a generator and when would you use one?"*
> A function that uses `yield` to return values lazily, one at a time, keeping its local state between calls. Use it for data larger than memory, infinite sequences (`itertools.count()`), and pipelines where each stage pulls from the previous. It is single-use: iterate twice and the second pass is empty.

## 4. Decorators: gift-wrap a function 🎁

**Analogy:** the restaurant cooks the biryani (your function). Swiggy does not touch the recipe — it *wraps* it: a tamper seal (validation), an insulated bag (caching), a "retry if the customer doesn't answer" policy. A decorator is a function that takes your function and returns a wrapped version. `@timer` above `def train()` is exactly `train = timer(train)`.

First, the two ingredients: **functions are objects** and **`*args, **kwargs`** pass anything through.

In [12]:
def shout(text):
    return text.upper() + "!"

alias = shout                    # a function is a value — no parentheses, not called
print(alias("hello"), "| name:", alias.__name__)

def show_all(*args, **kwargs):   # *args -> tuple of positionals, **kwargs -> dict of keywords
    print("args  =", args)
    print("kwargs=", kwargs)

show_all(1, "two", lr=0.01, epochs=5)
params = {"lr": 0.1, "epochs": 2}
show_all(*[7, 8], **params)      # the stars UNPACK on the way in

HELLO! | name: shout
args  = (1, 'two')
kwargs= {'lr': 0.01, 'epochs': 5}
args  = (7, 8)
kwargs= {'lr': 0.1, 'epochs': 2}


In [13]:
def timer(fn):
    """Decorator: print how long fn took. Works for ANY signature thanks to *args/**kwargs."""
    @functools.wraps(fn)                        # keep fn.__name__ / __doc__ on the wrapper
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        try:
            return fn(*args, **kwargs)          # call the original, pass everything through
        finally:                                # runs even if fn raises
            print(f"⏱ {fn.__name__} took {time.perf_counter() - t0:.4f}s")
    return wrapper

@timer                                          # same as: sum_squares = timer(sum_squares)
def sum_squares(n: int) -> int:
    """Sum of squares below n."""
    return sum(i * i for i in range(n))

print(sum_squares(1_000_000))
print("name preserved:", sum_squares.__name__, "| doc:", sum_squares.__doc__)

# Without @ syntax, the same thing:
def slow_hello(name, punctuation="!"):
    time.sleep(0.2); return f"hello {name}{punctuation}"
slow_hello = timer(slow_hello)
print(slow_hello("Kartikeya", punctuation="?"))

⏱ sum_squares took 0.0889s
333332833333500000
name preserved: sum_squares | doc: Sum of squares below n.
⏱ slow_hello took 0.2002s
hello Kartikeya?


### A decorator **with arguments** = one more layer
`@retry(times=3)` is a *call* that returns the real decorator. Three nested `def`s: `retry` (takes options) → `decorator` (takes the function) → `wrapper` (takes the call's arguments).

In [14]:
def retry(times=3, delay=0.2, exceptions=(Exception,)):
    def decorator(fn):                                   # the real decorator
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    if attempt == times:
                        raise                            # out of tries: re-raise the last error
                    print(f"  attempt {attempt} failed ({e}); retrying in {delay}s")
                    time.sleep(delay)
        return wrapper
    return decorator

calls = {"n": 0}

@timer                                                   # outer: times ALL attempts together
@retry(times=3, delay=0.1, exceptions=(ConnectionError,))
def call_llm_api(prompt: str) -> str:
    """Pretend network call that fails the first two times."""
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("timeout")
    return f"answer to {prompt!r}"

print(call_llm_api("What is a decorator?"))
print("total attempts:", calls["n"])

  attempt 1 failed (timeout); retrying in 0.1s
  attempt 2 failed (timeout); retrying in 0.1s
⏱ call_llm_api took 0.2004s
answer to 'What is a decorator?'
total attempts: 3


### `functools.lru_cache`: memoisation in one line
The naive recursive Fibonacci recomputes the same sub-problems billions of times. Cached, each `fib(k)` is computed once. Same decorator caches tokenizer loads, embedding lookups, expensive API calls — anything **pure** (same input → same output) with **hashable** arguments.

In [15]:
def fib_naive(n):
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

@functools.lru_cache(maxsize=None)
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

for n in (20, 25, 30):
    t0 = time.perf_counter(); fib_naive(n);  a = time.perf_counter() - t0
    fib_cached.cache_clear()
    t0 = time.perf_counter(); fib_cached(n); b = time.perf_counter() - t0
    print(f"fib({n}): naive {a*1000:8.2f} ms | cached {b*1000:8.4f} ms | speed-up {a/b:,.0f}x")

print(fib_cached(200))                       # instant; naive would take longer than the universe
print(fib_cached.cache_info())

fib(20): naive     2.76 ms | cached   0.0199 ms | speed-up 139x
fib(25): naive    31.01 ms | cached   0.0341 ms | speed-up 908x
fib(30): naive   323.08 ms | cached   0.0360 ms | speed-up 8,972x
280571172992510140037611932413038677189525
CacheInfo(hits=199, misses=201, maxsize=None, currsize=201)


## 5. Context managers: setup, work, guaranteed cleanup 🧹

`with X as y:` calls `X.__enter__()` (its return becomes `y`), runs your block, then calls `X.__exit__(...)` **no matter how the block ended** — even via an exception. That is why `with open()` never leaks files and why `with torch.no_grad():` reliably turns gradients back on.

Two ways to write one: a class with `__enter__/__exit__`, or a generator with `@contextmanager` (setup before `yield`, cleanup after).

In [16]:
class Timer:
    """Class-based context manager. Usage: with Timer('x') as t: ...; t.elapsed"""
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self                                   # becomes the `as` variable
    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.t0
        print(f"[{self.label}] {self.elapsed:.3f}s" + (f"  (exited via {exc_type.__name__})" if exc_type else ""))
        return False                                  # False = do NOT swallow the exception

@contextmanager
def timed(label):
    """Generator-based version: everything before yield = __enter__, after = __exit__."""
    t0 = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {time.perf_counter() - t0:.3f}s")

with Timer("sleep") as t:
    time.sleep(0.15)
print("captured:", round(t.elapsed, 2))

with timed("sum"):
    sum(range(2_000_000))

try:
    with timed("crash"):
        raise ValueError("boom")                      # cleanup still prints before the error propagates
except ValueError as e:
    print("caught after cleanup:", e)

[sleep] 0.150s
captured: 0.15
[sum] 0.087s
[crash] 0.000s
caught after cleanup: boom


In [17]:
# A realistic one: temporarily change a setting, and ALWAYS restore it (like torch.no_grad / torch.set_grad_enabled)
settings = {"grad": True}

@contextmanager
def no_grad():
    old = settings["grad"]
    settings["grad"] = False
    try:
        yield
    finally:
        settings["grad"] = old

print("before:", settings)
with no_grad():
    print("inside:", settings)
print("after: ", settings)

before: {'grad': True}
inside: {'grad': False}
after:  {'grad': True}


## 6. Dataclasses, type hints, `functools`, `collections` 🧰

**Type hints** don't change runtime behaviour — they are for humans and tools (autocomplete, Pyright/mypy, FastAPI validation). Modern syntax: `list[int]`, `int | None`.

**`@dataclass`** writes `__init__`, `__repr__`, `__eq__` for you from the annotated fields. Hugging Face's `TrainingArguments` is one giant dataclass.

In [18]:
@dataclass
class TrainConfig:
    lr: float = 1e-3
    epochs: int = 5
    batch_size: int = 32
    model: str = "resnet18"
    tags: list[str] = field(default_factory=list)   # NEVER `tags: list = []` -> shared mutable default!

cfg = TrainConfig(lr=3e-4, tags=["baseline"])
print(cfg)                                          # free __repr__
print(cfg == TrainConfig(lr=3e-4, tags=["baseline"]))   # free __eq__ (compares fields)
cfg.epochs = 10                                     # mutable by default
print("as dict:", cfg.__dict__)

@dataclass(frozen=True, order=True)                 # frozen -> immutable + hashable; order -> sortable
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
try:
    p.x = 5
except Exception as e:
    print(type(e).__name__, "-> frozen dataclasses cannot be changed")
print(sorted([Point(2, 1), Point(1, 9), Point(1, 2)]))
print("hashable, so usable as a dict key / cache key:", {p: "origin-ish"})

TrainConfig(lr=0.0003, epochs=5, batch_size=32, model='resnet18', tags=['baseline'])
True
as dict: {'lr': 0.0003, 'epochs': 10, 'batch_size': 32, 'model': 'resnet18', 'tags': ['baseline']}
FrozenInstanceError -> frozen dataclasses cannot be changed
[Point(x=1, y=2), Point(x=1, y=9), Point(x=2, y=1)]
hashable, so usable as a dict key / cache key: {Point(x=1.0, y=2.0): 'origin-ish'}


In [19]:
# Type hints: annotate, then let a tool check. They do NOT stop wrong calls at runtime.
def mean(xs: list[float]) -> float:
    """Average of a non-empty list."""
    return sum(xs) / len(xs)

def first_or_none(xs: list[int]) -> int | None:     # 3.10+ union syntax
    return xs[0] if xs else None

print(mean([1, 2, 3]), first_or_none([]), first_or_none([7]))
print("annotations are just metadata:", mean.__annotations__)
print(mean(["a", "b"]) if False else "a type checker (Pyright/mypy) would flag mean(['a','b']) before you run it")

2.0 None 7
annotations are just metadata: {'xs': list[float], 'return': <class 'float'>}
a type checker (Pyright/mypy) would flag mean(['a','b']) before you run it


In [20]:
# functools.partial: pre-fill some arguments -> a new function
def clip(x, lo, hi):
    return max(lo, min(hi, x))

relu6 = functools.partial(clip, lo=0, hi=6)
print([relu6(v) for v in (-3, 2, 9)])

# collections: the four you'll actually use
words = "the chai was hot the samosa was cold the service was fast".split()
print("Counter     :", Counter(words).most_common(3))

groups = defaultdict(list)                          # auto-creates [] on first access
for w in words:
    groups[len(w)].append(w)
print("defaultdict :", dict(groups))

Job = namedtuple("Job", "title city lpa")           # light immutable record with named fields
j = Job("ML Engineer", "Pune", 12.5)
print("namedtuple  :", j, j.lpa, j[0])

window = deque(maxlen=3)                            # fixed-size buffer: running average of last 3 losses
for loss in [0.9, 0.8, 0.7, 0.6, 0.5]:
    window.append(loss)
    print(f"   loss={loss}  last3={list(window)}  avg={sum(window)/len(window):.3f}")

[0, 2, 6]
Counter     : [('the', 3), ('was', 3), ('chai', 1)]
defaultdict : {3: ['the', 'was', 'hot', 'the', 'was', 'the', 'was'], 4: ['chai', 'cold', 'fast'], 6: ['samosa'], 7: ['service']}
namedtuple  : Job(title='ML Engineer', city='Pune', lpa=12.5) 12.5 ML Engineer
   loss=0.9  last3=[0.9]  avg=0.900
   loss=0.8  last3=[0.9, 0.8]  avg=0.850
   loss=0.7  last3=[0.9, 0.8, 0.7]  avg=0.800
   loss=0.6  last3=[0.8, 0.7, 0.6]  avg=0.700
   loss=0.5  last3=[0.7, 0.6, 0.5]  avg=0.600


## 7. 🎯 Try it yourself (in class, 5 minutes)

Write a decorator **`@log_calls`** that prints the function name, its positional args, keyword args and its return value on every call. Requirements:
- must work for *any* signature (`*args, **kwargs`),
- must keep the original name (`functools.wraps`),
- test it on two functions with different signatures.

In [24]:
def log_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"{fn.__name__}(args={args}, kwargs={kwargs}) -> {result!r}")
        return result
    return wrapper


@log_calls
def add(a, b):
    return a + b


@log_calls
def greet(name, excited=False):
    return f"hi {name}" + ("!" if excited else ".")


add(2, 3)
greet("Priya", excited=True)
print(add.__name__)


add(args=(2, 3), kwargs={}) -> 5
greet(args=('Priya',), kwargs={'excited': True}) -> 'hi Priya!'
add


<details><summary>Solution</summary>

```python
def log_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"{fn.__name__}(args={args}, kwargs={kwargs}) -> {result!r}")
        return result
    return wrapper

# add(args=(2, 3), kwargs={}) -> 5
# greet(args=('Priya',), kwargs={'excited': True}) -> 'hi Priya!'
# add
```
</details>

## Homework starter 🏠

- **Easy:** `@log_calls` (above) — finish it and add a `level="INFO"` argument (so it becomes a decorator *with* arguments).
- **Medium (job hunt):** `JobDataset` over yesterday's `jobs.csv` with `@dataclass Job`, `__len__`, `__getitem__`, `.filter(city=...)` returning a new dataset, and a generator method `.skills()` fed to `Counter`.
- **Stretch:** stream a 200 MB CSV through a generator pipeline in constant memory; verify with `tracemalloc`.

Scaffold for the Medium task (a small `jobs.csv` is generated so the cell runs standalone):

In [25]:
import csv
from collections.abc import Iterator

# tiny stand-in so the scaffold runs even without Day 01's file
with open("jobs.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["title", "company", "city", "salary_lpa", "skills"])
    w.writerows([["ML Engineer", "Zepto", "Bengaluru", "14.5", "python; pytorch; docker"],
                 ["Data Analyst", "TCS", "Pune", "5.5", "python; sql; excel"],
                 ["AI Engineer", "Razorpay", "Bengaluru", "18.0", "python; llms; langchain; aws"],
                 ["Data Analyst", "Infosys", "Pune", "Not disclosed", "sql; excel"]])

@dataclass(frozen=True)
class Job:
    title: str
    company: str
    city: str
    salary_lpa: float
    skills: frozenset[str]          # frozenset so the dataclass stays hashable

class JobDataset:
    def __init__(self, jobs: list[Job]):
        self.jobs = jobs

    @classmethod
    def from_csv(cls, path: str) -> "JobDataset":
        jobs = []
        with open(path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                try:
                    sal = float(r["salary_lpa"])
                except ValueError:
                    continue
                jobs.append(Job(r["title"], r["company"], r["city"], sal,
                                frozenset(s.strip().lower() for s in r["skills"].split(";"))))
        return cls(jobs)

    def __len__(self) -> int:
        return len(self.jobs)

    def __getitem__(self, i: int) -> Job:
        return self.jobs[i]

    def __repr__(self) -> str:
        return f"JobDataset(n={len(self)})"

    def filter(self, **conditions) -> "JobDataset":
        # YOUR CODE HERE: return a NEW JobDataset with jobs matching all conditions, e.g. filter(city="Pune")
        keep = [j for j in self.jobs if all(getattr(j, k) == v for k, v in conditions.items())]
        return JobDataset(keep)

    def skills(self) -> Iterator[str]:
        # YOUR CODE HERE: a generator that yields every skill of every job lazily
        for j in self.jobs:
            yield from j.skills            # `yield from` delegates to another iterable

jd = JobDataset.from_csv("jobs.csv")
print(jd, jd[0])
print(jd.filter(city="Bengaluru"), Counter(jd.skills()).most_common(3))

JobDataset(n=3) Job(title='ML Engineer', company='Zepto', city='Bengaluru', salary_lpa=14.5, skills=frozenset({'docker', 'python', 'pytorch'}))
JobDataset(n=2) [('python', 3), ('docker', 1), ('pytorch', 1)]


In [26]:
import random, tracemalloc, os
from collections import defaultdict

random.seed(42)

# ---- Step 1: generate a ~200MB CSV, written via a generator (never held in a list) ----
def fake_transactions(n):
    cities = ["Pune", "Delhi", "Mumbai", "Kochi", "Jaipur", "Bengaluru"]
    for _ in range(n):
        yield round(random.lognormvariate(6, 1), 2), random.choice(cities)

N_ROWS = 5_000_000   # tune this until the file lands near 200MB on your machine

with open("big_upi.csv", "w") as f:
    f.write("amount,city\n")
    for amt, city in fake_transactions(N_ROWS):
        f.write(f"{amt},{city}\n")

print(f"big_upi.csv: {os.path.getsize('big_upi.csv')/1e6:.1f} MB on disk")

# ---- Step 2: the streaming pipeline (same 3-stage shape as the notebook) ----
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        next(f)                          # skip header
        for line in f:
            yield line.rstrip("\n")

def parse(lines):
    for line in lines:
        amt, city = line.split(",")
        yield float(amt), city

def batched(items, size):
    batch = []
    for it in items:
        batch.append(it)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

# ---- Step 3: run it while tracking peak memory ----
tracemalloc.start()
pipeline = batched(parse(read_lines("big_upi.csv")), size=1000)

total, count, biggest = 0.0, 0, 0.0
per_city = defaultdict(float)

for batch in pipeline:
    for amt, city in batch:
        total += amt
        count += 1
        biggest = max(biggest, amt)
        per_city[city] += amt

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"rows processed = {count:,}")
print(f"mean = ₹{total/count:,.2f}   max = ₹{biggest:,.2f}")
print(f"peak RAM during streaming = {peak/1e6:.2f} MB")
print(f"file size on disk         = {os.path.getsize('big_upi.csv')/1e6:.1f} MB")
print("top city by volume:", max(per_city.items(), key=lambda kv: kv[1]))

big_upi.csv: 69.2 MB on disk
rows processed = 5,000,000
mean = ₹664.60   max = ₹51,184.88
peak RAM during streaming = 0.19 MB
file size on disk         = 69.2 MB
top city by volume: ('Pune', 554806328.950032)


## Key takeaways 🔑

- **Dunder methods** plug your object into Python syntax: `__len__` + `__getitem__` = a Dataset; `__call__` = a model; `__enter__/__exit__` = a context manager; `__repr__` = humane debugging.
- **`super().__init__()`** first in every subclass; prefer composition over deep inheritance.
- **Generators** pause at `yield` and keep their state; they stream data bigger than RAM, are single-use, and chain into pipelines (`batched(parse(read_lines(path)))`).
- **Decorators** = `fn = deco(fn)`. `*args/**kwargs` make them generic, `functools.wraps` keeps the name, one extra layer gives them arguments. `@lru_cache` = memoisation for free.
- **Context managers** guarantee cleanup even on exceptions; `@contextmanager` makes them from a generator.
- **Dataclasses + type hints** make configs self-documenting and tool-checkable; `field(default_factory=list)` avoids the shared-mutable-default bug.

**Before Day 03:** commit `ml_utils.py` (your `@timer`, `@retry`, `timed()`) to your `ai-ml-45days` repo and make sure `git --version` works on your laptop.